# Sparse PCA — warm-started from artificial patterns

Start from a loading pattern the desk *thinks* a factor should look
like (a 2s10s steepener mask, a butterfly, a short-end shock, …) and
let `factors.sparse_pca_warm` refine it against the actual cube. The
iteration has two opposing terms:

* a **data-fit** term that pulls loadings toward the top-variance
  direction of the panel (regular PCA);
* an **anchor** term `||w − w_prior||²` that pulls them back to the
  prior.

`anchor` is scaled by N internally so it's on a unit scale: `0` is
unconstrained PCA with a warm start, `~1` is balanced, `~10` is
essentially frozen. Optional `l1` adds explicit sparsification.

All analysis is on **daily diffs** of the wide panel — we hedge
moves, not levels.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pca import load_long, to_wide, EXPIRY_LABELS, TENOR_LABELS
from factors import sparse_pca_warm, loading_sparsity

sns.set_theme(style="whitegrid")

vol = to_wide(load_long("../data/mock/atm_vol.pkl")).diff().dropna()
print("vol diff:", vol.shape)

def loading_heatmap(ax, w_vec, title, vmax, cbar=False):
    series = pd.Series(w_vec, index=vol.columns)
    grid = series.unstack("tenor").reindex(
        index=[e for e in EXPIRY_LABELS if e in vol.columns.get_level_values("expiry")],
        columns=[t for t in TENOR_LABELS if t in vol.columns.get_level_values("tenor")],
    )
    sns.heatmap(grid, ax=ax, cmap="RdBu_r", center=0, vmin=-vmax, vmax=vmax, cbar=cbar)
    ax.set_title(title); ax.set_xlabel("Tenor"); ax.set_ylabel("Expiry")

## 1. Artificial pattern — 2s10s steepener

Across all expiries, weight the 2Y tenor at `−1` and the 10Y tenor at
`+1`. Everything else is zero. This is the canonical curve-steepener
mask — what a trader would write down before looking at any data.

In [ ]:
prior_2s10s = pd.Series(0.0, index=vol.columns, name="2s10s")
prior_2s10s[prior_2s10s.index.get_level_values("tenor") == "2Y"]  = -1.0
prior_2s10s[prior_2s10s.index.get_level_values("tenor") == "10Y"] = +1.0
prior_norm = prior_2s10s.values / np.linalg.norm(prior_2s10s.values)
print(f"prior nonzero cells: {(prior_2s10s != 0).sum()} / {len(prior_2s10s)}")

## 2. Anchor sweep — fit vs. prior similarity

Run `sparse_pca_warm` across a range of anchor weights. As anchor
rises, loadings move from the unconstrained top eigenvector
(`anchor=0`, cosine ≈ 0 with the prior) toward the prior itself
(`anchor≈20`, cosine ≈ 1). The variance share drops smoothly across
the trade-off.

In [ ]:
anchors = [0.0, 0.5, 2.0, 5.0, 20.0]
fits = {a: sparse_pca_warm(vol, prior_2s10s, anchor=a) for a in anchors}

rows = []
for a, (s, l, e) in fits.items():
    w = l.iloc[0].values
    rows.append({
        "anchor":         a,
        "var_explained":  float(e.iloc[0]),
        "||w - prior||":  float(np.linalg.norm(w - prior_norm)),
        "cos(w, prior)":  float(w @ prior_norm),
        "loading_gini":   float(loading_sparsity(l).iloc[0]),
    })
trade = pd.DataFrame(rows)
print(trade.round(4).to_string(index=False))

In [ ]:
# Loading heatmaps: prior + each anchored fit.
panels = [("prior (2s10s)", prior_norm)]
for a in anchors:
    panels.append((f"anchor={a}", fits[a][1].iloc[0].values))

vmax = max(float(np.abs(v).max()) for _, v in panels)
fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 5))
for ax, (title, w) in zip(axes, panels):
    loading_heatmap(ax, w, title, vmax)
plt.tight_layout(); plt.show()

# Trade-off curve.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(trade["cos(w, prior)"], trade["var_explained"], marker="o")
for _, r in trade.iterrows():
    ax.annotate(f"a={r['anchor']}", (r["cos(w, prior)"], r["var_explained"]),
                textcoords="offset points", xytext=(6, 4))
ax.set_xlabel("cos(w, prior) — prior similarity")
ax.set_ylabel("variance share")
ax.set_title("Anchor trade-off curve")
plt.tight_layout(); plt.show()

## 3. Adding L1 — explicit sparsification

At a fixed `anchor`, sweep `l1` to soft-threshold the loading entries.
`l1` is N-scaled so values in `[0, ~1]` are meaningful. Larger `l1`
kills more cells; the prior region (2Y, 10Y stripes) survives longest.

In [ ]:
l1_values = [0.0, 0.05, 0.2, 0.5]
l1_fits = {l1: sparse_pca_warm(vol, prior_2s10s, anchor=1.0, l1=l1) for l1 in l1_values}

rows = []
for l1, (s, l, e) in l1_fits.items():
    w = l.iloc[0].values
    rows.append({
        "l1":            l1,
        "var_explained": float(e.iloc[0]),
        "nonzero(>1e-3)": int((np.abs(w) > 1e-3).sum()),
        "loading_gini":  float(loading_sparsity(l).iloc[0]),
        "cos(w, prior)": float(w @ prior_norm),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))

panels = [(f"l1={l1}", l1_fits[l1][1].iloc[0].values) for l1 in l1_values]
vmax = max(float(np.abs(v).max()) for _, v in panels)
fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 5))
for ax, (title, w) in zip(axes, panels):
    loading_heatmap(ax, w, title, vmax)
plt.tight_layout(); plt.show()

## 4. Multiple priors — steepener + butterfly

Pass a multi-row prior to fit several anchored factors in one call.
Each row becomes one initialised factor; the algorithm deflates the
panel between factors. Useful when the desk has a small basis of
hand-drawn patterns and wants the data-refined version of each.

In [ ]:
prior_multi = pd.DataFrame(0.0, index=["2s10s", "5y_butterfly"], columns=vol.columns)
tenor = prior_multi.columns.get_level_values("tenor")
prior_multi.loc["2s10s", tenor == "2Y"]        = -1.0
prior_multi.loc["2s10s", tenor == "10Y"]       = +1.0
prior_multi.loc["5y_butterfly", tenor == "2Y"]  = -1.0
prior_multi.loc["5y_butterfly", tenor == "5Y"]  = +2.0
prior_multi.loc["5y_butterfly", tenor == "10Y"] = -1.0

s_m, l_m, e_m = sparse_pca_warm(vol, prior_multi, anchor=2.0)
print("variance share per anchored factor:")
print(e_m.round(4))

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
vmax = max(
    float(np.abs(prior_multi.loc[name].values / np.linalg.norm(prior_multi.loc[name].values)).max())
    for name in prior_multi.index
)
for j, name in enumerate(prior_multi.index):
    p = prior_multi.loc[name].values
    p_n = p / np.linalg.norm(p)
    loading_heatmap(axes[j, 0], p_n,            f"{name} — prior",        vmax)
    loading_heatmap(axes[j, 1], l_m.loc[name].values, f"{name} — anchored fit", vmax)
plt.tight_layout(); plt.show()

## 5. Using saved priors from the Streamlit app

The `pattern_creator.py` Streamlit app saves a `pd.DataFrame` to
`data/priors.pkl` whose index is pattern names and whose columns are
a `(expiry, tenor)` MultiIndex — exactly the multi-factor prior shape
`sparse_pca_warm` expects. Load and fit in one step.

Re-run the app (`streamlit run pattern_creator.py`) to overwrite the
pkl with a different set of patterns; this section needs no edits.

In [ ]:
from pathlib import Path

priors_path = Path("../data/priors.pkl")
if not priors_path.exists():
    print(f"No prior file at {priors_path.resolve()}. "
          f"Run `streamlit run pattern_creator.py` and click Save to create one.")
else:
    saved_priors = pd.read_pickle(priors_path)
    # Align to the diff panel's columns (Streamlit-saved priors use the
    # canonical universe from config.py, so this is usually a no-op).
    saved_priors = saved_priors.reindex(columns=vol.columns, fill_value=0.0)
    print(f"loaded {len(saved_priors)} priors from {priors_path}:")
    print("  ", list(saved_priors.index))

    # One anchored fit covering every saved pattern.
    s_saved, l_saved, e_saved = sparse_pca_warm(vol, saved_priors, anchor=2.0)

    rows = []
    for name in saved_priors.index:
        p = saved_priors.loc[name].values
        p_norm = p / (np.linalg.norm(p) or 1.0)
        w = l_saved.loc[name].values
        rows.append({
            "pattern":       name,
            "var_explained": float(e_saved.loc[name]),
            "cos(w, prior)": float(w @ p_norm),
            "loading_gini":  float(loading_sparsity(l_saved.loc[[name]]).iloc[0]),
        })
    summary = pd.DataFrame(rows)
    print()
    print(summary.round(4).to_string(index=False))

In [ ]:
# Side-by-side: each saved prior next to its anchored fit, on a shared
# colour scale per pattern. Sanity-check that the fit drifts toward
# market structure but stays recognisably the prior.
if priors_path.exists():
    n_p = len(saved_priors)
    fig, axes = plt.subplots(n_p, 2, figsize=(11, 4.5 * n_p),
                             squeeze=False)
    for j, name in enumerate(saved_priors.index):
        p = saved_priors.loc[name].values
        p_norm = p / (np.linalg.norm(p) or 1.0)
        w = l_saved.loc[name].values
        vmax = max(float(np.abs(p_norm).max()), float(np.abs(w).max()), 1e-9)
        loading_heatmap(axes[j, 0], p_norm, f"{name} — prior",        vmax)
        loading_heatmap(axes[j, 1], w,      f"{name} — anchored fit", vmax)
    plt.tight_layout(); plt.show()

## Notes

* Sections 1–5 use `sparse_pca_warm` — sequential power
  iteration with deflation, unit-norm loadings. Build priors in
  `streamlit_apps/pattern_creator.py`, load them in Section 5.
* For the joint-ALS variant (`soft_constrained_pca`) on the same
  priors, see `notebooks/soft_constrained_pca.ipynb`.